# 02 — Inference dengan YOLO Pretrained

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/03-object-detection/02_yolo_inference.ipynb)

**YOLO** (*You Only Look Once*) adalah keluarga detektor **one-stage**: satu kali *forward pass* langsung menghasilkan kotak, kelas, dan confidence. Karena itu YOLO sangat cepat dan populer untuk aplikasi real-time.

Kita memakai library [`ultralytics`](https://docs.ultralytics.com) dengan model **pretrained di dataset COCO** (80 kelas, misalnya orang, mobil, anjing, dan kursi).

Isi notebook:
1. Menjalankan deteksi pada beberapa gambar contoh.
2. Membedah objek hasil (`Results`): koordinat kotak, kelas, dan confidence.
3. Menggambar bounding box sendiri dengan Matplotlib.
4. Efek parameter `conf`, `iou` (NMS), dan filter kelas.
5. Membandingkan YOLO11n dengan YOLO26n.

> ⚖️ **Lisensi:** `ultralytics` berlisensi **AGPL-3.0**.

## 0. Cek GPU

Di Colab pilih **Runtime → Change runtime type → T4 GPU**. Inference YOLO tetap bisa berjalan di CPU (model `n` cukup ringan), tetapi GPU jauh lebih cepat, terutama untuk training.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, akan berjalan di CPU (lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Instalasi

`ultralytics` belum terpasang di Colab secara default. Perhatikan lisensinya: **AGPL-3.0**. Bebas dipakai untuk belajar dan proyek open-source, tetapi jika dipakai di produk/layanan yang tidak dibuka kodenya, perlu lisensi Enterprise dari Ultralytics.

In [ ]:
%pip install -q ultralytics

## 2. Import dan Gambar Contoh

Library `ultralytics` sudah menyertakan dua gambar contoh (`bus.jpg` dan `zidane.jpg`) di folder `ASSETS`, jadi tidak perlu diunduh. Kita juga mencoba mengunduh satu gambar dari dataset COCO. Jika gagal (misalnya tidak ada internet), notebook tetap lanjut dengan dua gambar bawaan.

In [ ]:
import time
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
from ultralytics.utils import ASSETS

gambar = [ASSETS / "bus.jpg", ASSETS / "zidane.jpg"]

# Gambar tambahan dari COCO val2017 (dua kucing di sofa)
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
tujuan = Path("kucing.jpg")
try:
    if not tujuan.exists():
        urllib.request.urlretrieve(url, tujuan)
    gambar.append(tujuan)
except Exception as e:
    print("Gagal mengunduh gambar tambahan, lanjut tanpa gambar ini:", e)

for g in gambar:
    print(g.name, Image.open(g).size)

## 3. Memuat Model

Nama file menentukan versi dan ukuran model: `yolo11n.pt` = YOLO11 ukuran **n**ano (paling kecil dan cepat). Ukuran lain: `s`, `m`, `l`, `x` (makin besar makin akurat tetapi makin lambat). Bobot diunduh otomatis saat pertama kali dipakai.

Kita mulai dengan **YOLO11**, yang di tahap *post-processing* memakai **NMS**, sama seperti yang kita implementasikan di notebook 01.

In [ ]:
model = YOLO("yolo11n.pt")

print("Jumlah kelas:", len(model.names))
print("Contoh kelas:", {i: model.names[i] for i in [0, 1, 2, 3, 5, 15, 16, 56, 62]})

## 4. Menjalankan Deteksi

`model.predict()` menerima path gambar, URL, array NumPy, atau list berisi semuanya. Hasilnya adalah list objek `Results`, satu per gambar.

Parameter penting:
- `conf`: ambang confidence minimum (default 0.25).
- `iou`: ambang IoU untuk NMS (default 0.7).
- `imgsz`: ukuran gambar input model (default 640).
- `verbose=False`: agar output tidak terlalu ramai.

`r.plot()` menghasilkan gambar dengan kotak sudah tergambar, dalam format **BGR** (konvensi OpenCV). Kita balik ke RGB dengan `[..., ::-1]` sebelum ditampilkan dengan Matplotlib.

In [ ]:
results = model.predict(gambar, conf=0.25, verbose=False)

fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, results):
    ax.imshow(r.plot()[..., ::-1])  # BGR -> RGB
    ax.set_title(f"{Path(r.path).name}: {len(r.boxes)} objek")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Membedah Objek `Results`

Atribut `r.boxes` berisi semua deteksi dalam satu gambar:

| Atribut | Isi |
|---|---|
| `boxes.xyxy` | koordinat piksel `x1, y1, x2, y2` |
| `boxes.xywhn` | format YOLO: pusat & ukuran ternormalisasi (0–1) |
| `boxes.conf` | confidence setiap kotak |
| `boxes.cls` | indeks kelas (float), dipetakan ke nama lewat `r.names` |

Semua berupa tensor PyTorch; gunakan `.cpu().numpy()` untuk mengubahnya ke NumPy.

In [ ]:
r = results[0]  # bus.jpg
print("Ukuran gambar asli (h, w):", r.orig_shape)
print("Kecepatan (ms):", {k: round(v, 1) for k, v in r.speed.items()})
print()

xyxy = r.boxes.xyxy.cpu().numpy()
xywhn = r.boxes.xywhn.cpu().numpy()
conf = r.boxes.conf.cpu().numpy()
cls = r.boxes.cls.cpu().numpy().astype(int)

print(f"{'kelas':<10} {'conf':>5}   {'x1':>6} {'y1':>6} {'x2':>6} {'y2':>6}   {'cx':>5} {'cy':>5} {'w':>5} {'h':>5}")
for k, c_, b, n in zip(cls, conf, xyxy, xywhn):
    print(f"{r.names[k]:<10} {c_:>5.2f}   {b[0]:>6.0f} {b[1]:>6.0f} {b[2]:>6.0f} {b[3]:>6.0f}   "
          f"{n[0]:>5.2f} {n[1]:>5.2f} {n[2]:>5.2f} {n[3]:>5.2f}")

## 6. Menggambar Bounding Box Sendiri

`r.plot()` praktis, tetapi dalam aplikasi nyata kita sering perlu menggambar sendiri (warna per kelas, hanya kelas tertentu, dan sebagainya). Cukup pakai koordinat `xyxy` dan `matplotlib.patches.Rectangle`, persis seperti di notebook 01.

In [ ]:
def tampilkan_deteksi(r, ax, min_conf=0.0):
    ax.imshow(Image.open(r.path).convert("RGB"))
    warna = plt.cm.tab10.colors
    for b, c_, k in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(),
                        r.boxes.cls.cpu().numpy().astype(int)):
        if c_ < min_conf:
            continue
        x1, y1, x2, y2 = b
        col = warna[k % 10]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=col, lw=2))
        ax.text(x1, y1, f"{r.names[k]} {c_:.2f}", color="white", fontsize=9,
                va="bottom", bbox=dict(facecolor=col, pad=1, edgecolor="none"))
    ax.axis("off")


fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6))
for ax, r in zip(np.atleast_1d(axes), results):
    tampilkan_deteksi(r, ax)
    ax.set_title(Path(r.path).name)
plt.tight_layout()
plt.show()

## 7. Efek Parameter `conf` dan `iou`

### 7.1 Ambang confidence

`conf` rendah → lebih banyak objek terdeteksi (recall naik), tetapi lebih banyak deteksi salah (precision turun). Ini trade-off precision–recall dari notebook 01.

In [ ]:
for c_thr in [0.05, 0.1, 0.25, 0.5, 0.75]:
    r = model.predict(gambar[0], conf=c_thr, verbose=False)[0]
    nama = [r.names[int(k)] for k in r.boxes.cls]
    ringkas = {n: nama.count(n) for n in sorted(set(nama))}
    print(f"conf ≥ {c_thr:<4} → {len(r.boxes):2d} kotak {ringkas}")

### 7.2 Ambang IoU untuk NMS

Dengan `conf` rendah (banyak kandidat), kita ubah `iou`. Ambang tinggi membiarkan lebih banyak kotak yang saling tumpang tindih lolos (potensi duplikat), sedangkan ambang rendah menekan lebih agresif.

In [ ]:
for iou_thr in [0.3, 0.5, 0.7, 0.9]:
    r = model.predict(gambar[0], conf=0.05, iou=iou_thr, verbose=False)[0]
    print(f"iou = {iou_thr} → {len(r.boxes):2d} kotak")

fig, axes = plt.subplots(1, 2, figsize=(12, 7))
for ax, iou_thr in zip(axes, [0.3, 0.9]):
    r = model.predict(gambar[0], conf=0.05, iou=iou_thr, verbose=False)[0]
    tampilkan_deteksi(r, ax)
    ax.set_title(f"conf=0.05, iou={iou_thr} → {len(r.boxes)} kotak")
plt.tight_layout()
plt.show()

### 7.3 Filter kelas

Parameter `classes` membatasi deteksi ke kelas tertentu, misalnya `classes=[0]` untuk hanya mendeteksi **orang**. Ini berguna untuk aplikasi seperti menghitung orang.

In [ ]:
for r in model.predict(gambar, classes=[0], verbose=False):
    print(f"{Path(r.path).name:<12} jumlah orang: {len(r.boxes)}")

## 8. Perbandingan YOLO11n vs YOLO26n

**YOLO26** adalah generasi yang lebih baru di `ultralytics`. Menurut dokumentasi Ultralytics, YOLO26 dirancang *end-to-end* sehingga **tidak memerlukan NMS** di tahap post-processing. Itu salah satu alasan post-processing-nya lebih cepat, terutama di CPU dan perangkat edge.

Kita bandingkan jumlah deteksi dan waktu, dengan waktu *inference* (forward pass model) dan *postprocess* (termasuk NMS pada YOLO11) dipisah. Satu kali prediksi awal (*warm-up*) dijalankan dulu agar waktu inisialisasi tidak ikut terhitung.

Waktu sangat bergantung pada hardware (GPU/CPU, ukuran batch, dan ukuran gambar), jadi jangan heran jika selisihnya kecil atau bahkan terbalik untuk satu gambar. Perbedaan biasanya lebih terasa saat gambar berisi banyak objek atau di perangkat edge.

In [ ]:
def ukur(model, sumber, ulang=5):
    model.predict(sumber, verbose=False)  # warm-up
    inf, post = [], []
    for _ in range(ulang):
        r = model.predict(sumber, verbose=False)[0]
        inf.append(r.speed["inference"])
        post.append(r.speed["postprocess"])
    return r, float(np.mean(inf)), float(np.mean(post))


model26 = YOLO("yolo26n.pt")
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
for ax, (nama, m) in zip(axes, [("YOLO11n", model), ("YOLO26n", model26)]):
    r, t_inf, t_post = ukur(m, gambar[0])
    n_param = sum(p.numel() for p in m.model.parameters())
    print(f"{nama}: {len(r.boxes)} objek | inference {t_inf:.1f} ms | postprocess {t_post:.1f} ms | "
          f"{n_param / 1e6:.1f} juta parameter")
    tampilkan_deteksi(r, ax)
    ax.set_title(f"{nama}: {len(r.boxes)} objek, {t_inf + t_post:.0f} ms")
plt.tight_layout()
plt.show()

## Ringkasan

- `YOLO("yolo11n.pt")` memuat model pretrained COCO (80 kelas); `model.predict()` mengembalikan list `Results`.
- `r.boxes.xyxy`, `.xywhn`, `.conf`, `.cls` adalah bahan untuk logika aplikasi apa pun (menghitung, memfilter, menggambar).
- `conf` mengatur trade-off precision–recall; `iou` mengatur agresivitas NMS; `classes` memfilter kelas.
- Model yang lebih besar (`s`, `m`, …) lebih akurat tetapi lebih lambat, jadi pilih sesuai kebutuhan latensi.
- Ingat lisensi **AGPL-3.0** jika ingin memakai `ultralytics` di produk.

## Latihan

1. **Gambar sendiri.** Upload 3 foto milikmu ke Colab (`from google.colab import files; files.upload()`). Jalankan deteksi dan cari minimal satu kasus gagal: objek tidak terdeteksi, salah kelas, atau kotak ganda. Jelaskan kemungkinan penyebabnya (ukuran objek, pencahayaan, kelas tidak ada di COCO, dan sebagainya).
2. **Penghitung objek.** Tulis fungsi `hitung_objek(path, conf=0.4)` yang mengembalikan dictionary `{nama_kelas: jumlah}` dan juga menyimpan gambar hasil anotasi ke file (petunjuk: `r.save(filename=...)`). Uji pada semua gambar di notebook ini.
3. **Ukuran model vs kecepatan.** Bandingkan `yolo11n.pt`, `yolo11s.pt`, dan `yolo11m.pt` pada gambar `bus.jpg` di GPU dan di CPU (`device="cpu"`): catat waktu inference, jumlah deteksi, dan rata-rata confidence. Buat grafik batang dan simpulkan model mana yang cocok untuk (a) kamera CCTV real-time dan (b) analisis foto offline.